# 04 Gold - Telecommunications

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Builds the industry KPIs from accepted Silver records.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; it is not an importable module.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name)
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

if re.fullmatch(r"u_[0-9a-f]{16}", participant_key) is None:
    raise ValueError("Invalid participant_key")
if lab_id != 'telecommunications':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

from pyspark.sql import functions as F

silver = {"network_sites": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/network_sites/", "plans": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/plans/", "subscribers": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/subscribers/", "usage_events": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/usage_events/"}
gold = {"site_daily": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/telecommunications/telecommunications_site_daily/", "subscriber_monthly": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/telecommunications/telecommunications_subscriber_monthly/"}
plans = spark.read.format("delta").load(silver["plans"])
sites = spark.read.format("delta").load(silver["network_sites"])
subscribers = spark.read.format("delta").load(silver["subscribers"])
usage = spark.read.format("delta").load(silver["usage_events"])
subscriber_monthly = (usage.join(
    subscribers.select("participant_key", "subscriber_id", "plan_id"),
    ["participant_key", "subscriber_id"],
)
    .withColumn("month", F.trunc("event_time", "month"))
    .groupBy("participant_key", "month", "subscriber_id", "plan_id")
    .agg(F.sum(F.when(F.col("usage_type") == "data", F.col("usage_value")).otherwise(0)).alias("data_mb"), F.sum(F.when(F.col("usage_type") == "voice", F.col("usage_value")).otherwise(0)).alias("voice_minutes"), F.sum(F.when(F.col("usage_type") == "sms", F.col("usage_value")).otherwise(0)).alias("sms_count"), F.sum("charge_amount").alias("usage_charge"))
    .withColumn("overage_flag", F.col("usage_charge") > 0))
subscriber_monthly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold["subscriber_monthly"])
site_daily = (usage.join(
    sites.select("participant_key", "site_id", "capacity_mb_day"),
    ["participant_key", "site_id"],
)
    .withColumn("event_date", F.to_date("event_time"))
    .groupBy("participant_key", "event_date", "site_id", "capacity_mb_day")
    .agg(F.countDistinct("subscriber_id").alias("unique_subscribers"), F.sum(F.when(F.col("usage_type") == "data", F.col("usage_value")).otherwise(0)).alias("data_mb"), F.sum(F.when(F.col("usage_type") == "voice", F.col("usage_value")).otherwise(0)).alias("voice_minutes"))
    .withColumn("utilization_pct", F.round(F.col("data_mb") / F.col("capacity_mb_day") * 100, 2)).drop("capacity_mb_day"))
site_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold["site_daily"])
subscriber_monthly.show(20, truncate=False)

for table_name, location in gold.items():
    row_count = spark.read.format("delta").load(location).count()
    assert row_count > 0, f"Gold table {table_name} is empty"
    print(f"Gold {table_name}: {row_count} rows")

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_gold.{participant_key}_telecommunications_subscriber_monthly (`participant_key` STRING, `month` DATE, `subscriber_id` STRING, `plan_id` STRING, `data_mb` DOUBLE, `voice_minutes` DOUBLE, `sms_count` DOUBLE, `usage_charge` DOUBLE, `overage_flag` BOOLEAN) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/telecommunications/telecommunications_subscriber_monthly/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_gold.{participant_key}_telecommunications_site_daily (`participant_key` STRING, `event_date` DATE, `site_id` STRING, `unique_subscribers` BIGINT, `data_mb` DOUBLE, `voice_minutes` DOUBLE, `utilization_pct` DOUBLE) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/telecommunications/telecommunications_site_daily/'""")


## Expected result

Two non-empty, industry-specific aggregate Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
